# Search-term regression: search-trends second column versus GP admissions

This notebook uses the search-term files already present in the repository. They are expected to have filenames beginning with `time_series_GB`.

The goal is to regress GP admissions against the **second column** of each search-trends file, regardless of the column name. The selected second column is converted to a canonical `count` field inside the analysis code.

## Model

The first-pass aggregate model is:

```text
z(GP admissions)_t = beta_0 + beta_1 z(search count)_t + beta_2 z(search count)_{t-1} + ... + seasonality + error_t
```

Here `search count` is the sum of the second source column across all selected `time_series_GB*` files at each time point. Later, set `AGGREGATE_TERMS = False` to model individual search terms separately, still using each file's second column.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from wastewater.io import find_repo_root
from wastewater.search_terms import (
    find_search_term_files,
    build_search_term_catalogue,
    search_file_to_long,
    build_search_term_regression_frame,
    fit_search_term_ols,
)
from wastewater.ukhsa import build_ukhsa_series_catalogue

ROOT = find_repo_root(ROOT)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
ROOT

## 1. Find search-term files

This scans the whole checkout for files beginning with `time_series_GB`.

In [ ]:
search_files = find_search_term_files(ROOT)
display(search_files.sort_values('path'))
print(f'Found {len(search_files)} search-term files')

## 2. Inspect inferred schemas

The important check is the `predictor_column` column below. It should be the second source column in each search-trends file. Python uses zero-based indexing, so `VALUE_COLUMN_INDEX = 1` means the second column.

In [ ]:
VALUE_COLUMN_INDEX = 1  # second source column

search_catalogue = build_search_term_catalogue(ROOT, value_column_index=VALUE_COLUMN_INDEX)
display(search_catalogue[['path', 'filename', 'date_column', 'predictor_column_index', 'predictor_column', 'usable_predictor_values', 'status', 'error']].sort_values('path'))

# Full column list for debugging.
display(search_catalogue[['path', 'columns']])

## 3. Choose GP admissions outcome files

The outcome is taken from the UKHSA chart files classified as GP/admissions. If inference is wrong, manually set `OUTCOME_FILES` using paths from the catalogue.

In [ ]:
ukhsa_catalogue = build_ukhsa_series_catalogue(ROOT)
display(ukhsa_catalogue[['path', 'filename', 'series_type', 'date_column', 'value_column']].sort_values(['series_type', 'path']))

SEARCH_FILES = search_catalogue.loc[search_catalogue['status'] == 'ok', 'path'].tolist()
OUTCOME_FILES = ukhsa_catalogue.loc[ukhsa_catalogue['series_type'] == 'gp_admissions', 'path'].tolist()

# Manual override examples:
# SEARCH_FILES = ['data/raw/time_series_GB_...csv']
# OUTCOME_FILES = ['data/raw/ukhsa-chart-...gp-admissions...csv']

print('Search files using second source column:')
for path in SEARCH_FILES:
    print(' -', path)

print('\nGP admissions outcome files:')
for path in OUTCOME_FILES:
    print(' -', path)

## 4. Preview search-term second-column series

In [ ]:
for path in SEARCH_FILES[:5]:
    print('\nSearch file:', path)
    display(search_file_to_long(ROOT, path, value_column_index=VALUE_COLUMN_INDEX).head())

## 5. Build regression frame

Use weekly aggregation first. Switch `FREQ = 'M'` if the search-term and GP-admission series are monthly. The predictor is always built from the second source column in each search-trends file.

In [ ]:
FREQ = 'W'
LAGS = [0, 1, 2, 3, 4]
AGGREGATE_TERMS = True

regression_frame = build_search_term_regression_frame(
    ROOT,
    search_files=SEARCH_FILES,
    outcome_files=OUTCOME_FILES,
    freq=FREQ,
    lags=LAGS,
    aggregate_terms=AGGREGATE_TERMS,
    value_column_index=VALUE_COLUMN_INDEX,
)

display(regression_frame)

out = PROCESSED / 'search_terms_second_column_gp_admissions_regression_frame.csv'
regression_frame.to_csv(out, index=False)
print(f'Saved {len(regression_frame):,} rows to {out.relative_to(ROOT)}')

## 6. Plot aligned series

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(regression_frame['period'], regression_frame['z_search_count'], label='Search-trends second column, z-scored')
ax.plot(regression_frame['period'], regression_frame['z_gp_admissions'], label='GP admissions, z-scored')
ax.set_title('Search-trends second column and GP admissions')
ax.set_xlabel('Period')
ax.set_ylabel('z-score')
ax.legend()
fig.autofmt_xdate()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(regression_frame['z_search_count'], regression_frame['z_gp_admissions'])
ax.set_xlabel('Search-trends second column, z-scored')
ax.set_ylabel('GP admissions, z-scored')
ax.set_title('Contemporaneous association')
plt.show()

## 7. Fit lagged regression

The model uses HAC standard errors to reduce sensitivity to residual autocorrelation.

In [ ]:
result = fit_search_term_ols(
    regression_frame,
    lags=LAGS,
    predictor_prefix='z_search_count_lag',
    seasonal_controls=True,
)
print(result.summary())

## 8. Observed versus fitted

In [ ]:
predictor_cols = [f'z_search_count_lag{lag}' for lag in LAGS]
model_rows = regression_frame.dropna(subset=['z_gp_admissions', *predictor_cols]).copy()
model_rows['prediction'] = result.predict()
model_rows['residual'] = model_rows['z_gp_admissions'] - model_rows['prediction']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_rows['period'], model_rows['z_gp_admissions'], label='Observed GP admissions, z')
ax.plot(model_rows['period'], model_rows['prediction'], label='Predicted GP admissions, z')
ax.set_title('Observed versus fitted GP admissions from search-trends second column')
ax.legend()
fig.autofmt_xdate()
plt.show()

model_rows.to_csv(PROCESSED / 'search_terms_second_column_gp_admissions_model_rows.csv', index=False)
model_rows[['period', 'z_gp_admissions', 'prediction', 'residual']].head()

## 9. Next refinements

- Set `AGGREGATE_TERMS = False` to inspect individual search-term predictors, still using each file's second column.
- Try a distributed lag model with more weeks of lag.
- Compare weekly and monthly aggregation.
- Add UKHSA NHS 111 call series and wastewater predictors after the search-term baseline is stable.
- Use train/test splits or rolling-origin validation before treating this as predictive.